# Import libraries - Import các thư viện

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

## Tải các thư viện

In [3]:
# !pip install gensim

In [4]:
# !pip install fasttext

In [5]:
# !pip install pyvi

In [6]:
!pip install vncorenlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 17.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for vncorenlp: filename=vncorenlp-1.0.3-py3-none-any.whl size=2645933 sha256=fb5db54ff83af65dce71cad188a03b22e8e8d465b1dc14988eb6a717372dbae0
  Stored in directory: /root/.cache/pip/wheels/6f/19/20/ec7083125fd06db1a19d0d3ca18806ecf4e8ed1464713b4efa
Successfully built vncorenlp


## Import thư viện

In [7]:
import pandas as pd
import numpy as np
import re
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, confusion_matrix
# from pyvi import ViTokenizer
from vncorenlp import VnCoreNLP
# import fasttext
# import gensim

## Preprocessing data - Tiền xử lý dữ liệu

## Load data - Load dữ liệu

In [8]:
df = pd.read_csv('https://raw.githubusercontent.com/PTIT-Projects/nlp-assignment-web-scraping/refs/heads/main/reviews7188_exception.csv')
df.head()

,data,numStars,label
0,1 điểm cho thợ lắp. Tận tâm mà chả tận tâm tí nào,1,0
1,"Máy lạnh mình mua tháng 6 năm ngoái, dùng cũng...",1,0
2,Không biết ĐMX Hội Ăn lắp máy cũ của khách trả...,1,0
3,"Máy lấp ngày 6/7/25 dàn lạnh ok, êm, dàn nóng ...",3,1
4,Đổi máy lần 2 vẫn là hàng có vấn đề. Cục nóng ...,1,0


In [9]:
df['label'].value_counts()

,count
label,
2,3751
0,2186
1,1251


In [10]:
df['label'].value_counts(normalize=True)

,proportion
label,
2,0.521842
0,0.304118
1,0.174040


Nhận xét: Trong dataset cào được, tỉ lệ review hài lòng - không hài lòng là xấp xỉ 80% hài lòng và 20% không hài lòng

In [11]:
#Xây dựng dict về các từ teencode và viết tắt chuyển thành từ chuẩn
teencode_url = 'https://raw.githubusercontent.com/PTIT-Projects/nlp-assignment-web-scraping/main/teencode.txt'

teencode_dict = {}

response = requests.get(teencode_url)
teencode_data = response.text

for line in teencode_data.split('\n'):
    if '\t' in line:
        key, value = line.split('\t', 1)
        teencode_dict[key.strip()] = value.strip()

for i, (key, value) in enumerate(teencode_dict.items()):
    if i >= 5:
        break
    print(f"'{key}': '{value}'")

'ctrai': 'con trai'
'khôg': 'không'
'bme': 'bố mẹ'
'nv': 'nhân viên'
'cskh': 'chăm sóc khách hàng'


In [12]:
#Xóa từ dừng
stopwords_url = 'https://raw.githubusercontent.com/stopwords/vietnamese-stopwords/refs/heads/master/vietnamese-stopwords.txt'

response = requests.get(stopwords_url)
stopwords = set(response.text.splitlines())

In [13]:
def to_lowercase(text):
  text = str(text)
  return text.lower()
# Xóa HTML tags
def remove_html_tags(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)
def remove_urls_emails_phones(text):
    # Xóa URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Xóa email
    text = re.sub(r'\S+@\S+', '', text)
    # Xóa SDT
    text = re.sub(r'\d{10,11}', '', text)
    return text
#Xóa kí tự đặc biệt
def remove_special_characters(text):
  text = re.sub(r'[^a-z0-9A-Zàáâãèéêìíòóôõùúăđĩũơưăạảấầẩẫậắằẳẵặẹẻẽếềểễệỉịọỏốồổỗộớờởỡợụủứừửữựỳýỵỷỹ\s]', ' ', text)
  text = re.sub(r'[0-9]', '', text)
  return text
#thay các từ teencode, viết tắt thành từ chuẩn
def normalize_teencode(text, teencode_dict):
    words = text.split()
    normalized_words = []
    for word in words:
        if word in teencode_dict:
            normalized_words.append(teencode_dict[word])
        else:
            normalized_words.append(word)
    return ' '.join(normalized_words)
def normalize_word_overall(text, teencode_dict):
  new_text = to_lowercase(text)
  new_text = remove_html_tags(new_text)
  new_text = remove_urls_emails_phones(new_text)
  new_text = remove_special_characters(new_text)
  new_text = normalize_teencode(new_text, teencode_dict)
  return new_text

In [14]:
df['data'] = df['data'].apply(lambda x: normalize_word_overall(x, teencode_dict))

In [15]:
pd.set_option('display.max_colwidth', None)
display(df['data'].head(10))

,data
0,điểm cho thợ lắp tận tâm mà chả tận tâm tí nào
1,máy lạnh mình mua tháng năm ngoái dùng cũng ít năm nay chưa mở máy ngày nào đến hôm nay mở máy thì không lên nguồn quá tệ
2,không biết điện máy xanh hội ăn lắp máy cũ của khách trả lại hay sao mà khi giao máy thấy dàn lạnh dán băng kéo tùm lum lá nhôm có dấu hiệu móp méo giống đồ cũ của khách hàng đổi trả chạy mấy ngày thì thấy trong dàn nóng phun ra một đống hạt llâms tấm đen như phân của con gián
3,máy lấp ngày dàn lạnh ok êm dàn nóng bị rỉ nước gọi tổng đài bh điện máy xanh bảo máy lăp tháng bị v là bthuong máy dòng khác nhà mình không bị v tình trạng v có bình thường không mọi người
4,đổi máy lần vẫn là hàng có vấn đề cục nóng cục lạnh kêu rung tường ồn không ngủ được mới đổi máy
5,rất tệ quá sức tệ đây là lần thứ mua lại hãng panasonic trải nghiệm cực kì tệ từ chất lượng đến chăm sóc viên máy mới mua bật lần đầu luôn không mát chỉ có luồn gió và hơi lạnh theo hướng thẳng bên nóng thứ máy chạy chế độ quiet êm ái không quạt mà kêu rền rền ù ù nhức hết cả đâu gây khó chịu khi ngủ phản hồi bên kỹ thuậtphòng m trong khi thực tế chỉ m phòng to không làm lạnh đủ hp chỉ làm lạnh được m trong khi thông số đăng bán là từ m panasonic chăm sóc khách hàng gọi đánh giá dịch vụ mình đáng giá không hài lòng thì kêu mình tự gọi lên tổng đài vậy gọi gỏi ý kiến đánh giá để làm gì
6,cục nóng kêu to cần khắc phục
7,cục lạnh giờ ồn quá ồn hãng vào bảo hành hoài tình trạng máy không hết lỗi đề nghị cửa hàng làm việc lại với hãng và tôi muốn đổi sản phẩm khác không theo dõi thêm nữa hãng trì truệ kéo dài thời gian quá trời có cả video sản phẩm ồn mà hãng cứ bát bỏ không giải quyết
8,quá tệ cục nóng kêu rất to tôi muốn đổi sản phẩm khác
9,mới mua tr nay giảm còn tr


In [16]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

In [17]:
# vncorenlp = VnCoreNLP("/content/drive/MyDrive/Colab Notebooks/PTIT/NLP/VnCoreNLP/VnCoreNLP-1.1.1.jar", annotators="wseg", max_heap_size = '-Xmx500m')
import os

vncorenlp_dir = '/content/VnCoreNLP'
if not os.path.exists(vncorenlp_dir):
    print("Downloading VnCoreNLP models...")
    !git clone https://github.com/vncorenlp/VnCoreNLP.git /content/VnCoreNLP

# Initialize the VNcoreNLP parser
vncorenlp_path = os.path.join(vncorenlp_dir, 'VnCoreNLP-1.2.jar')
vncorenlp = VnCoreNLP(vncorenlp_path, annotators="wseg")

Cloning into '/content/VnCoreNLP'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 259 (delta 17), reused 33 (delta 11), pack-reused 212 (from 1)
Receiving objects: 100% (259/259), 237.79 MiB | 15.54 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Updating files: 100% (34/34), done.


In [18]:
# result_list = []
# for text in df['data']:
#   result = vncorenlp.annotate(text)
#   tokenize_result = result['sentences']
#   tokenize_result_list = []
#   for result in tokenize_result:
#       for word in result:
#           s = word['form'].replace('_', ' ')
#           tokenize_result_list.append(s)
#   result_list.append(tokenize_result_list)

In [19]:
def custom_tokenizer_vncorenlp(text):
  result = vncorenlp.annotate(text)
  tokenize_result = result['sentences']
  tokenize_result_list = []
  for result in tokenize_result:
      for word in result:
          s = word['form'].replace('_', ' ')
          if (s in stopwords):
            continue
          tokenize_result_list.append(s)
  return tokenize_result_list

In [20]:
# # convert to custom tokenizer for tf-idf
# def custom_tokenizer_vncorenlp(text):
#     list_of_tokens = ViTokenizer.tokenize(text)
#     new_token_lists = list_of_tokens.split()
#     result_list = [word.replace('_', ' ') for word in new_token_lists]
#     result_list = [word for word in result_list if word not in stopwords]
#     return result_list

# Trích chọn đặc trưng

## TF-IDF

In [21]:
# tf_idf_vncorenlp = TfidfVectorizer(tokenizer=custom_tokenizer_vncorenlp)
tf_idf = TfidfVectorizer(analyzer = 'word', tokenizer=custom_tokenizer_vncorenlp, token_pattern = None)

In [22]:
X_tdidf = tf_idf.fit_transform(df['data'].astype(str))
y_tfidf = df['label']

In [23]:
X_tdidf.shape

(7188, 5123)

## Bag of Word

In [24]:
count_vectorizer = CountVectorizer(analyzer = 'word', tokenizer=custom_tokenizer_vncorenlp, token_pattern = None)
X_bow = count_vectorizer.fit_transform(df['data'].astype(str))
y_bow = df['label']

In [25]:
X_bow.shape

(7188, 5123)

# Xây dựng và huấn luyện mô hình

## Chia tập dữ liệu

In [26]:
X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test = train_test_split(X_tdidf, y_tfidf, test_size=0.2, random_state=42)
X_bow_train, X_bow_test, y_bow_train, y_bow_test = train_test_split(X_bow, y_bow, test_size=0.2, random_state=42)

## Sử dụng Naive Bayes

In [27]:
def train_and_evaluate(model, X_train, X_test, y_train, y_test, name="Model"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n=== {name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print("Confusion Matrix:\n", cm)
    return acc, prec, rec, f1, cm

### Khi áp dụng TF-IDF để trích chọn đặc trưng

In [28]:

# naive bayes
# nb_tfidf = MultinomialNB()
nb_tfidf = MultinomialNB(alpha=1.0, fit_prior=True)
train_and_evaluate(nb_tfidf, X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test, name="Naive Bayes (TF-IDF)")


=== Naive Bayes (TF-IDF) ===
Accuracy : 0.7594
Precision: 0.7644
Recall   : 0.7594
F1-Score : 0.7009
Confusion Matrix:
 [[340   4  60]
 [126  18 120]
 [ 35   1 734]]


(0.7593880389429764,
 0.7643527221719199,
 0.7593880389429764,
 0.7009096995983451,
 array([[340,   4,  60],
        [126,  18, 120],
        [ 35,   1, 734]]))

### Khi sử dụng BoW để trích chọn đặc trưng

In [29]:
# boW
# nb_bow = MultinomialNB()
nb_bow = MultinomialNB(alpha=1.0, fit_prior=True)
train_and_evaluate(nb_bow, X_bow_train, X_bow_test, y_bow_train, y_bow_test, name="Naive Bayes (BoW)")


=== Naive Bayes (BoW) ===
Accuracy : 0.7698
Precision: 0.7475
Recall   : 0.7698
F1-Score : 0.7524
Confusion Matrix:
 [[315  50  39]
 [ 94  79  91]
 [ 30  27 713]]


(0.7698191933240612,
 0.7474518960177569,
 0.7698191933240612,
 0.7524113699624289,
 array([[315,  50,  39],
        [ 94,  79,  91],
        [ 30,  27, 713]]))

## Sử dụng SVM

### Khi áp dụng TF-IDF để trích chọn đặc trưng

In [30]:
# # TF-IDF
# svm_tfidf = SVC()
svm_tfidf = SVC( kernel="rbf", C=0.5, probability=False)

train_and_evaluate(svm_tfidf, X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test, name="SVM (TF-IDF)")


=== SVM (TF-IDF) ===
Accuracy : 0.7809
Precision: 0.7757
Recall   : 0.7809
F1-Score : 0.7449
Confusion Matrix:
 [[364  11  29]
 [133  46  85]
 [ 47  10 713]]


(0.7809457579972183,
 0.7756846845296226,
 0.7809457579972183,
 0.7449054440919547,
 array([[364,  11,  29],
        [133,  46,  85],
        [ 47,  10, 713]]))

### Khi sử dụng BoW để trích chọn đặc trưng

In [31]:
# boW
# svm_bow = SVC()
svm_bow = SVC( kernel="rbf", C=0.5, probability=False)
train_and_evaluate(svm_bow, X_bow_train, X_bow_test, y_bow_train, y_bow_test, name="SVM (BoW)")


=== SVM (BoW) ===
Accuracy : 0.7663
Precision: 0.7503
Recall   : 0.7663
F1-Score : 0.7360
Confusion Matrix:
 [[331  24  49]
 [107  56 101]
 [ 46   9 715]]


(0.7663421418636995,
 0.7502611990182303,
 0.7663421418636995,
 0.7360207770045839,
 array([[331,  24,  49],
        [107,  56, 101],
        [ 46,   9, 715]]))

### So sánh 2 thuật toán với 2 phương thức TF-IDF và BoW

In [32]:
results = {}

# Naive Bayes - Bow
results["Naive Bayes - Bow"] = train_and_evaluate(
    nb_bow, X_bow_train, X_bow_test, y_bow_train, y_bow_test, name="Naive Bayes (BoW)"
)

# Naive Bayes - TF-IDF
results["Naive Bayes - TF-IDF"] = train_and_evaluate(
    nb_tfidf, X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test, name="Naive Bayes (TF-IDF)"
)

# SVM - BoW
results["SVM - BoW"] = train_and_evaluate(
    svm_bow, X_bow_train, X_bow_test, y_bow_train, y_bow_test, name="SVM (BoW)"
)

# SVM - TF-IDF
results["SVM - TF-IDF"] = train_and_evaluate(
    svm_tfidf, X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test, name="SVM (TF-IDF)"
)

# Tạo DataFrame gồm cả confusion matrix
df_new = pd.DataFrame.from_dict(
    results,
    orient="index",
    columns=["Accuracy", "Precision", "Recall", "F1-Score", "Confusion Matrix"]
)
df_new


=== Naive Bayes (BoW) ===
Accuracy : 0.7698
Precision: 0.7475
Recall   : 0.7698
F1-Score : 0.7524
Confusion Matrix:
 [[315  50  39]
 [ 94  79  91]
 [ 30  27 713]]

=== Naive Bayes (TF-IDF) ===
Accuracy : 0.7594
Precision: 0.7644
Recall   : 0.7594
F1-Score : 0.7009
Confusion Matrix:
 [[340   4  60]
 [126  18 120]
 [ 35   1 734]]

=== SVM (BoW) ===
Accuracy : 0.7663
Precision: 0.7503
Recall   : 0.7663
F1-Score : 0.7360
Confusion Matrix:
 [[331  24  49]
 [107  56 101]
 [ 46   9 715]]

=== SVM (TF-IDF) ===
Accuracy : 0.7809
Precision: 0.7757
Recall   : 0.7809
F1-Score : 0.7449
Confusion Matrix:
 [[364  11  29]
 [133  46  85]
 [ 47  10 713]]


,Accuracy,Precision,Recall,F1-Score,Confusion Matrix
Naive Bayes - Bow,0.769819,0.747452,0.769819,0.752411,"[[315, 50, 39], [94, 79, 91], [30, 27, 713]]"
Naive Bayes - TF-IDF,0.759388,0.764353,0.759388,0.700910,"[[340, 4, 60], [126, 18, 120], [35, 1, 734]]"
SVM - BoW,0.766342,0.750261,0.766342,0.736021,"[[331, 24, 49], [107, 56, 101], [46, 9, 715]]"
SVM - TF-IDF,0.780946,0.775685,0.780946,0.744905,"[[364, 11, 29], [133, 46, 85], [47, 10, 713]]"


## Test thử với 3 bình luận có 3 nhãn khác nhau

In [33]:
test_data = pd.DataFrame({
    'data': [
        "Vừa mua xong mà bực cả mình. Van hơi thì rò. Ko tích áp hơi",
        "Đa năng, dễ sử dụng",
        "Bình thường"
    ],
    'label': [0, 2, 1]
})
display(test_data)

,data,label
0,Vừa mua xong mà bực cả mình. Van hơi thì rò. Ko tích áp hơi,0
1,"Đa năng, dễ sử dụng",2
2,Bình thường,1


In [34]:
test_data['data'] = test_data['data'].apply(lambda x: normalize_word_overall(x, teencode_dict))

In [35]:
test_data['data']

,data
0,vừa mua xong mà bực cả mình van hơi thì rò không tích áp hơi
1,đa năng dễ sử dụng
2,bình thường


### Sử dụng TF-IDF

#### Mô hình Naive Bayes

In [36]:
def predict_test_data(df, vectorizer, model):
  data_vector = vectorizer.transform(df['data'])
  predictions = model.predict(data_vector)
  return predictions, df['label'].to_list()

In [37]:
predict_test_data(test_data, tf_idf, nb_tfidf)

(array([0, 2, 2]), [0, 2, 1])

#### Mô hình SVM

In [38]:
predict_test_data(test_data, tf_idf, svm_tfidf)

(array([2, 2, 1]), [0, 2, 1])

### Sử dụng BoW

#### Mô hình Naive Bayes

In [39]:
predict_test_data(test_data, count_vectorizer, nb_bow)

(array([0, 2, 2]), [0, 2, 1])

#### Mô hình SVM

In [40]:
predict_test_data(test_data, count_vectorizer, svm_bow)

(array([1, 2, 2]), [0, 2, 1])

# StreamLit

In [41]:
%%writefile demo.py
import pandas as pd
import numpy as np
import re
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, confusion_matrix
# from pyvi import ViTokenizer
from vncorenlp import VnCoreNLP
import os

df = pd.read_csv('https://raw.githubusercontent.com/PTIT-Projects/nlp-assignment-web-scraping/refs/heads/main/reviews7188_exception.csv')
# df.head()

df['label'].value_counts()
df['label'].value_counts(normalize=True)
teencode_url = 'https://raw.githubusercontent.com/PTIT-Projects/nlp-assignment-web-scraping/main/teencode.txt'

teencode_dict = {}

response = requests.get(teencode_url)
teencode_data = response.text

for line in teencode_data.split('\n'):
    if '\t' in line:
        key, value = line.split('\t', 1)
        teencode_dict[key.strip()] = value.strip()

for i, (key, value) in enumerate(teencode_dict.items()):
    if i >= 5:
        break
    print(f"'{key}': '{value}'")

stopwords_url = 'https://raw.githubusercontent.com/stopwords/vietnamese-stopwords/refs/heads/master/vietnamese-stopwords.txt'

response = requests.get(stopwords_url)
stopwords = set(response.text.splitlines())


def to_lowercase(text):
  text = str(text)
  return text.lower()
# Xóa HTML tags
def remove_html_tags(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)
def remove_urls_emails_phones(text):
    # Xóa URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Xóa email
    text = re.sub(r'\S+@\S+', '', text)
    # Xóa SDT
    text = re.sub(r'\d{10,11}', '', text)
    return text
#Xóa kí tự đặc biệt
def remove_special_characters(text):
  text = re.sub(r'[^a-z0-9A-Zàáâãèéêìíòóôõùúăđĩũơưăạảấầẩẫậắằẳẵặẹẻẽếềểễệỉịọỏốồổỗộớờởỡợụủứừửữựỳýỵỷỹ\s]', ' ', text)
  text = re.sub(r'[0-9]', '', text)
  return text
#thay các từ teencode, viết tắt thành từ chuẩn
def normalize_teencode(text, teencode_dict):
    words = text.split()
    normalized_words = []
    for word in words:
        if word in teencode_dict:
            normalized_words.append(teencode_dict[word])
        else:
            normalized_words.append(word)
    return ' '.join(normalized_words)
def normalize_word_overall(text, teencode_dict):
  new_text = to_lowercase(text)
  new_text = remove_html_tags(new_text)
  new_text = remove_urls_emails_phones(new_text)
  new_text = remove_special_characters(new_text)
  new_text = normalize_teencode(new_text, teencode_dict)
  return new_text

df['data'] = df['data'].apply(lambda x: normalize_word_overall(x, teencode_dict))
pd.set_option('display.max_colwidth', None)

vncorenlp_dir = '/content/VnCoreNLP'
if not os.path.exists(vncorenlp_dir):
    print("Downloading VnCoreNLP models...")
    # !git clone https://github.com/vncorenlp/VnCoreNLP.git /content/VnCoreNLP
    subprocess.run(["git", "clone", "https://github.com/vncorenlp/VnCoreNLP.git", vncorenlp_dir])

# Initialize the VNcoreNLP parser
vncorenlp_path = os.path.join(vncorenlp_dir, 'VnCoreNLP-1.2.jar')
vncorenlp = VnCoreNLP(vncorenlp_path, annotators="wseg")

def custom_tokenizer_vncorenlp(text):
  result = vncorenlp.annotate(text)
  tokenize_result = result['sentences']
  tokenize_result_list = []
  for result in tokenize_result:
      for word in result:
          s = word['form'].replace('_', ' ')
          if (s in stopwords):
            continue
          tokenize_result_list.append(s)
  return tokenize_result_list

tf_idf = TfidfVectorizer(analyzer = 'word', tokenizer=custom_tokenizer_vncorenlp, token_pattern = None)
X_tdidf = tf_idf.fit_transform(df['data'].astype(str))
y_tfidf = df['label']

count_vectorizer = CountVectorizer(analyzer = 'word', tokenizer=custom_tokenizer_vncorenlp, token_pattern = None)
X_bow = count_vectorizer.fit_transform(df['data'].astype(str))
y_bow = df['label']

X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test = train_test_split(X_tdidf, y_tfidf, test_size=0.2, random_state=42)
X_bow_train, X_bow_test, y_bow_train, y_bow_test = train_test_split(X_bow, y_bow, test_size=0.2, random_state=42)

def train_and_evaluate(model, X_train, X_test, y_train, y_test, name="Model"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n=== {name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print("Confusion Matrix:\n", cm)
    return acc, prec, rec, f1, cm

nb_tfidf = MultinomialNB(alpha=1.0, fit_prior=True)
nb_bow = MultinomialNB(alpha=1.0, fit_prior=True)
svm_tfidf = SVC( kernel="rbf", C=1.0, probability=False)
svm_bow = SVC( kernel="rbf", C=1.0, probability=False)
results = {}

# Naive Bayes - Bow
results["Naive Bayes - Bow"] = train_and_evaluate(
    nb_bow, X_bow_train, X_bow_test, y_bow_train, y_bow_test, name="Naive Bayes (BoW)"
)

# Naive Bayes - TF-IDF
results["Naive Bayes - TF-IDF"] = train_and_evaluate(
    nb_tfidf, X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test, name="Naive Bayes (TF-IDF)"
)

# SVM - BoW
results["SVM - BoW"] = train_and_evaluate(
    svm_bow, X_bow_train, X_bow_test, y_bow_train, y_bow_test, name="SVM (BoW)"
)

# SVM - TF-IDF
results["SVM - TF-IDF"] = train_and_evaluate(
    svm_tfidf, X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test, name="SVM (TF-IDF)"
)

# Tạo DataFrame gồm cả confusion matrix
df_new = pd.DataFrame.from_dict(
    results,
    orient="index",
    columns=["Accuracy", "Precision", "Recall", "F1-Score", "Confusion Matrix"]
)

test_data = pd.DataFrame({
    'data': [
        "Vừa mua xong mà bực cả mình. Van hơi thì rò. Ko tích áp hơi",
        "Đa năng, dễ sử dụng",
        "Bình thường"
    ],
    'label': [0, 2, 1]
})
display(test_data)

test_data['data'] = test_data['data'].apply(lambda x: normalize_word_overall(x, teencode_dict))

test_data['data']

def predict_test_data(df, vectorizer, model):
  data_vector = vectorizer.transform(df['data'])
  predictions = model.predict(data_vector)
  return predictions, df['label'].to_list()


#=============================Giao diện Streamlit ======
import streamlit as st

st.title("Phân loại bình luận")
st.markdown("Phân loại bình luận : tốt/ bình thg / không tốt")

text = st.text_area("✍️ Nhập bình luận:", height=250)

if st.button("Đánh giá"):
    if not text.strip():
        st.warning("⚠️ Vui lòng nhập nội dung.")
    else:
        with st.spinner("🔄 Đang xử lý"):



Writing demo.py


In [42]:
!pip install -q streamlit==1.41.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 75.4 MB/s eta 0:00:00


In [43]:
!wget -q -O - https://loca.lt/mytunnelpassword

35.229.222.102

In [44]:
# run streamlit through port: 8501
!streamlit run demo.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧⠇⠏⠋
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.229.222.102:8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y)   Stopping...
^C
